In [10]:
import sys
!{sys.executable} -m pip install pymupdf chromadb openai anthropic langchain langchain-text-splitters tiktoken scikit-learn numpy supabase tavily-python

  Using cached langchain_text_splitters-1.1.2-py3-none-any.whl.metadata (3.3 kB)
  Using cached tiktoken-0.13.0-cp313-cp313-win_amd64.whl.metadata (6.8 kB)
  Using cached scikit_learn-1.9.0-cp313-cp313-win_amd64.whl.metadata (11 kB)
  Using cached supabase-2.31.0-py3-none-any.whl.metadata (4.7 kB)
  Using cached distro-1.9.0-py3-none-any.whl.metadata (6.8 kB)
  Using cached jiter-0.15.0-cp313-cp313-win_amd64.whl.metadata (5.3 kB)
  Using cached sniffio-1.3.1-py3-none-any.whl.metadata (3.9 kB)
  Using cached docstring_parser-0.18.0-py3-none-any.whl.metadata (3.5 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached narwhals-2.22.1-py3-none-any.whl.metadata (15 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
  Using cached realtime-2.31.0-py3-none-any.whl.metadata (7.0 kB)
  Using cached supabase_functions-2.31.0-py3-none-any.whl.metadata (2.4 kB)
  Using cached storage3-2.31.0-py3-none-any.whl.metadata (2.2 kB)
  Using cached supabase

In [5]:
import chromadb
client = chromadb.PersistentClient(path="./chroma_db")
collection = client.get_or_create_collection("financial_rag")

In [ ]:
client.delete_collection("financial_rag")
collection = client.get_or_create_collection(
    name="financial_rag",
    metadata={"hnsw:space": "cosine"}    # ChromaDB defaults to L2 (Euclidean) distance if you not specified. Options: "cosine", "l2" (default), "ip" (inner product).
)
print(collection.count())       # expecting 0 — empty, correctly recreated
print(collection.metadata)      # expect {'hnsw:space': 'cosine'}

0
{'hnsw:space': 'cosine'}


In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()   #loading .env file

anthropic_key = os.getenv("ANTHROPIC_API_KEY")      #reads env variables
openai_key = os.getenv("OPENAI_API_KEY")

# load_dotenv() → reads .env file → injects into os.environ
# os.getenv("KEY") → retrieves from os.environ

# os.environ is the in-memory store sitting between the two.
# the in-memory dictionary holding the values

In [ ]:
def web_search_fallback(question):
    print(f"  Web search fallback for: {question}")
    
    results = tavily_client.search(
        query=question + " 2026",
        search_depth="advanced",
        max_results=3
    )
    
    context = ""
    sources = []
    for r in results["results"]:
        context += f"[Source: {r['url']}]\n{r['content']}\n\n"
        sources.append(("Web", r['url'][:50]))
    
    response = client_anthropic.messages.create(
        model="claude-sonnet-4-5",
        max_tokens=800,
        messages=[{"role": "user", "content": f"""Answer the following question using the web search results below.
        The current year is 2026. Use the most recent data available in the results.
        Cite each fact with the source URL.
        If sources conflict, use the most recent one.

Search results:
{context}

Question: {question}"""}]
    )
    
    return {
        "answer": response.content[0].text,
        "confidence": 0.5,
        "query_type": "web_search",
        "sources": sources
    }

In [11]:
import pymupdf
import json
import os

# map filename keywords to bank names
BANK_NAMES = {
    "hdfc": "HDFC Bank",
    "icici": "ICICI Bank",
    "sbi": "SBI",
    "axis": "Axis Bank",
    "kotak": "Kotak Mahindra Bank"
}

def get_bank_name(filename):
    filename_lower = filename.lower()
    for keyword, bank_name in BANK_NAMES.items():
        if keyword in filename_lower:
            return bank_name
    return "Unknown"

def parse_pdfs(pdf_folder, output_path):
    all_pages = []
    
    for filename in os.listdir(pdf_folder):
        if not filename.endswith(".pdf"):
            continue
            
        bank_name = get_bank_name(filename)
        pdf_path = os.path.join(pdf_folder, filename)
        
        print(f"Parsing {bank_name}...")
        
        doc = pymupdf.open(pdf_path)
        
        for page_num, page in enumerate(doc, start=1):
            text = page.get_text()
            
            # skip blank or near-blank pages
            if len(text.strip()) < 50:
                continue
            
            all_pages.append({
                "bank_name": bank_name,
                "page_number": page_num,
                "fiscal_year": "FY25",
                "text": text
            })
        
        print(f"  → {len(doc)} pages processed")
        doc.close()
    
    # save to json
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(all_pages, f, ensure_ascii=False, indent=2)
    
    print(f"\nDone. {len(all_pages)} pages saved to {output_path}")

# run it
parse_pdfs("data/pdfs", "data/parsed_pages.json")

Parsing Axis Bank...
  → 504 pages processed
Parsing HDFC Bank...
  → 590 pages processed
Parsing ICICI Bank...
  → 341 pages processed
Parsing Kotak Mahindra Bank...
  → 522 pages processed
Parsing SBI...
  → 517 pages processed

Done. 2460 pages saved to data/parsed_pages.json


In [ ]:
# # peek at one page to make sure data looks right
# with open("data/parsed_pages.json", "r", encoding="utf-8") as f:
#     pages = json.load(f)

# # print first page
# print(pages[0])

{'bank_name': 'Axis Bank', 'page_number': 2, 'fiscal_year': 'FY25', 'text': 'Inside this report\nReporting context\n4\t\nAbout the report\nWelcome to Axis Bank\n10\t\nAbout Axis Bank\n12\t Integrated business lines\n14\t\nChairman’s statement\n16\t\nStrategic pillars\n18\t\nAdvancing our ESG agenda\n20\t Presence\n22\t Milestones\n24\t\nOwnership structure\n26 \nOperating landscape\n30 Marketing Initiatives\n40\t Board of directors\n41\t\nCore management team\nValue creation by the bank\n44 Value creation model\n48\t Stakeholder engagement\n58\t Materiality assessment\nBusiness performance review\n70\t MD & CEO’s statement \n76\t\nExternal environment  \n84 Strategy in action\n88\t Key performance indicators  \n92\t Message from the management – Retail banking  \n96\t\nBusiness segment performance – Retail banking  \n102\t Message from the management – Wholesale banking \n104\t Business segment performance – Wholesale banking  \n108 ‘One Axis’ in action \n116 \x07Message from the manag

In [12]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
import json

def chunk_pages(input_path, output_path):
    # load parsed pages
    with open(input_path, "r", encoding="utf-8") as f:
        pages = json.load(f)
    
    splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name="cl100k_base",
    chunk_size=500,
    chunk_overlap=50,
)
    
    all_chunks = []
    chunk_id = 0
    
    for page in pages:
        # skip if text too short
        if len(page["text"].strip()) < 50:
            continue
        
        chunks = splitter.split_text(page["text"])
        
        for chunk in chunks:
            all_chunks.append({
                "chunk_id": f"chunk_{chunk_id}",
                "bank_name": page["bank_name"],
                "page_number": page["page_number"],
                "fiscal_year": page["fiscal_year"],
                "text": chunk,
                "level": "leaf"
            })
            chunk_id += 1
    
    # save to json
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(all_chunks, f, ensure_ascii=False, indent=2)
    
    print(f"Done. {len(all_chunks)} chunks saved to {output_path}")

# run it
chunk_pages("data/parsed_pages.json", "data/chunked_data.json")

Done. 5496 chunks saved to data/chunked_data.json


In [ ]:
# # peek at one chunk
# with open("data/chunked_data.json", "r", encoding="utf-8") as f:
#     chunks = json.load(f)

# print(f"Total chunks: {len(chunks)}")
# print(f"\nSample chunk:")
# print(chunks[0])

Total chunks: 5496

Sample chunk:
{'chunk_id': 'chunk_0', 'bank_name': 'Axis Bank', 'page_number': 2, 'fiscal_year': 'FY25', 'text': 'Inside this report\nReporting context\n4\t\nAbout the report\nWelcome to Axis Bank\n10\t\nAbout Axis Bank\n12\t Integrated business lines\n14\t\nChairman’s statement\n16\t\nStrategic pillars\n18\t\nAdvancing our ESG agenda\n20\t Presence\n22\t Milestones\n24\t\nOwnership structure\n26 \nOperating landscape\n30 Marketing Initiatives\n40\t Board of directors\n41\t\nCore management team\nValue creation by the bank\n44 Value creation model\n48\t Stakeholder engagement\n58\t Materiality assessment\nBusiness performance review\n70\t MD & CEO’s statement \n76\t\nExternal environment  \n84 Strategy in action\n88\t Key performance indicators  \n92\t Message from the management – Retail banking  \n96\t\nBusiness segment performance – Retail banking  \n102\t Message from the management – Wholesale banking \n104\t Business segment performance – Wholesale banking  \n

In [15]:
from openai import OpenAI
import json
import os
import time
from dotenv import load_dotenv

load_dotenv()

client_openai = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# ── delete stale embeddings file if chunk count has changed ──
if os.path.exists("data/chunks_with_embeddings.json"):
    with open("data/chunks_with_embeddings.json", "r", encoding="utf-8") as f:
        existing_check = json.load(f)
    with open("data/chunked_data.json", "r", encoding="utf-8") as f:
        current_check = json.load(f)
    if len(existing_check) != len(current_check):
        os.remove("data/chunks_with_embeddings.json")
        print(f"Chunk count changed ({len(existing_check)} → {len(current_check)}). Deleted stale embeddings file. Starting fresh.")
    else:
        print(f"Chunk count matches ({len(current_check)}). Resume logic active.")

def embed_texts_with_retry(texts, batch_size=50, max_retries=3):
    all_embeddings = []
    
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        
        for attempt in range(max_retries):
            try:
                response = client_openai.embeddings.create(
                    input=batch,
                    model="text-embedding-3-small"
                )
                batch_embeddings = [item.embedding for item in response.data]
                all_embeddings.extend(batch_embeddings)
                print(f"  Embedded {min(i+batch_size, len(texts))}/{len(texts)} chunks...")
                break
            except Exception as e:
                if attempt < max_retries - 1:
                    print(f"  Error on batch {i} — retrying in 5 seconds... ({e})")
                    time.sleep(5)
                else:
                    print(f"  Failed after {max_retries} attempts. Saving progress...")
                    for j, chunk in enumerate(chunks[:len(all_embeddings)]):
                        chunk["embedding"] = all_embeddings[j]
                    with open("data/chunks_with_embeddings.json", "w", encoding="utf-8") as f:
                        json.dump(chunks[:len(all_embeddings)], f, ensure_ascii=False, indent=2)
                    print(f"  Saved {len(all_embeddings)} chunks. Re-run to continue.")
                    raise
    
    return all_embeddings

# load chunks
with open("data/chunked_data.json", "r", encoding="utf-8") as f:
    chunks = json.load(f)

# check if we already have partial embeddings
already_done = 0
if os.path.exists("data/chunks_with_embeddings.json"):
    with open("data/chunks_with_embeddings.json", "r", encoding="utf-8") as f:
        existing = json.load(f)
    already_done = len(existing)
    print(f"Resuming from chunk {already_done}...")
    for i in range(already_done):
        chunks[i]["embedding"] = existing[i]["embedding"]

# only embed remaining chunks
remaining_texts = [chunk["text"] for chunk in chunks[already_done:]]

if len(remaining_texts) == 0:
    print("All chunks already embedded!")
else:
    print(f"Embedding {len(remaining_texts)} remaining chunks...")
    new_embeddings = embed_texts_with_retry(remaining_texts)
    
    for i, embedding in enumerate(new_embeddings):
        chunks[already_done + i]["embedding"] = embedding

    with open("data/chunks_with_embeddings.json", "w", encoding="utf-8") as f:
        json.dump(chunks, f, ensure_ascii=False, indent=2)

    print(f"\nDone. All {len(chunks)} chunks embedded and saved.")

Chunk count changed (11579 → 5496). Deleted stale embeddings file. Starting fresh.
Embedding 5496 remaining chunks...
  Embedded 50/5496 chunks...
  Embedded 100/5496 chunks...
  Embedded 150/5496 chunks...
  Embedded 200/5496 chunks...
  Embedded 250/5496 chunks...
  Embedded 300/5496 chunks...
  Embedded 350/5496 chunks...
  Embedded 400/5496 chunks...
  Embedded 450/5496 chunks...
  Embedded 500/5496 chunks...
  Embedded 550/5496 chunks...
  Embedded 600/5496 chunks...
  Embedded 650/5496 chunks...
  Embedded 700/5496 chunks...
  Embedded 750/5496 chunks...
  Embedded 800/5496 chunks...
  Embedded 850/5496 chunks...
  Embedded 900/5496 chunks...
  Embedded 950/5496 chunks...
  Embedded 1000/5496 chunks...
  Embedded 1050/5496 chunks...
  Embedded 1100/5496 chunks...
  Embedded 1150/5496 chunks...
  Embedded 1200/5496 chunks...
  Embedded 1250/5496 chunks...
  Embedded 1300/5496 chunks...
  Embedded 1350/5496 chunks...
  Embedded 1400/5496 chunks...
  Embedded 1450/5496 chunks...
  E

In [17]:
# # run this to check exact token count
# import json
# import tiktoken

# encoder = tiktoken.encoding_for_model("text-embedding-3-small")

# with open("data/chunked_data.json", "r", encoding="utf-8") as f:
#     chunks = json.load(f)

# total_tokens = sum(len(encoder.encode(chunk["text"])) for chunk in chunks)

# print(f"Total tokens: {total_tokens:,}")
# print(f"Estimated cost: ${total_tokens / 1_000_000 * 0.02:.4f}")

In [18]:
import chromadb
import json

# connect to chromadb
client_chroma = chromadb.PersistentClient(path="./chroma_db")
collection = client_chroma.get_or_create_collection(
    name="financial_rag",
    metadata={"hnsw:space": "cosine"}
)

# load chunks with embeddings
with open("data/chunks_with_embeddings.json", "r", encoding="utf-8") as f:
    chunks = json.load(f)

# upload in batches of 100
batch_size = 100
total = len(chunks)

for i in range(0, total, batch_size):
    batch = chunks[i:i+batch_size]
    
    collection.add(
        ids=[c["chunk_id"] for c in batch],
        embeddings=[c["embedding"] for c in batch],
        documents=[c["text"] for c in batch],
        metadatas=[{
            "bank_name": c["bank_name"],
            "page_number": c["page_number"],
            "fiscal_year": c["fiscal_year"],
            "level": c["level"]
        } for c in batch]
    )
    
    print(f"Uploaded {min(i+batch_size, total)}/{total} chunks...")

print(f"\nDone. {collection.count()} chunks in ChromaDB.")

Uploaded 100/5496 chunks...
Uploaded 200/5496 chunks...
Uploaded 300/5496 chunks...
Uploaded 400/5496 chunks...
Uploaded 500/5496 chunks...
Uploaded 600/5496 chunks...
Uploaded 700/5496 chunks...
Uploaded 800/5496 chunks...
Uploaded 900/5496 chunks...
Uploaded 1000/5496 chunks...
Uploaded 1100/5496 chunks...
Uploaded 1200/5496 chunks...
Uploaded 1300/5496 chunks...
Uploaded 1400/5496 chunks...
Uploaded 1500/5496 chunks...
Uploaded 1600/5496 chunks...
Uploaded 1700/5496 chunks...
Uploaded 1800/5496 chunks...
Uploaded 1900/5496 chunks...
Uploaded 2000/5496 chunks...
Uploaded 2100/5496 chunks...
Uploaded 2200/5496 chunks...
Uploaded 2300/5496 chunks...
Uploaded 2400/5496 chunks...
Uploaded 2500/5496 chunks...
Uploaded 2600/5496 chunks...
Uploaded 2700/5496 chunks...
Uploaded 2800/5496 chunks...
Uploaded 2900/5496 chunks...
Uploaded 3000/5496 chunks...
Uploaded 3100/5496 chunks...
Uploaded 3200/5496 chunks...
Uploaded 3300/5496 chunks...
Uploaded 3400/5496 chunks...
Uploaded 3500/5496 chun

In [19]:
# test a simple query
query = "What is the NPA ratio of HDFC Bank?"

# embed the query
query_embedding = client_openai.embeddings.create(
    input=query,
    model="text-embedding-3-small"
).data[0].embedding

# search chromadb
results = collection.query(
    query_embeddings=[query_embedding],
    n_results=3,
    include=["documents", "metadatas", "distances"]
)

# print results
for i in range(3):
    print(f"\n--- Result {i+1} ---")
    print(f"Bank: {results['metadatas'][0][i]['bank_name']}")
    print(f"Page: {results['metadatas'][0][i]['page_number']}")
    print(f"Distance: {results['distances'][0][i]:.4f}")
    print(f"Text: {results['documents'][0][i][:200]}...")


--- Result 1 ---
Bank: HDFC Bank
Page: 38
Distance: 0.3270
Text: per share rose by 12.8 per cent to J22.0 in FY25.
23,79,786 
18,83,395 
27,14,715 
FY24
FY25
FY23
Deposits (I Cr)
27,14,715 
40.2
40.4
40.5 
FY24
FY25
FY23
Cost to Income Ratio (%)
40.5  
1.24
1.12 
1...

--- Result 2 ---
Bank: Kotak Mahindra Bank
Page: 122
Distance: 0.3425
Text: Net NPA (H crore)
 3,106 
 2,149 
 1,479 
 1,567 
 1,752 
Gross NPA Ratio
3.2%
2.4%
1.8%
1.4%
1.4%
Net NPA Ratio
1.2%
0.7%
0.4%
0.4%
0.4%
           (H in crore)
MARKET RELATED RATIOS 
 FY 2020-21
 FY...

--- Result 3 ---
Bank: HDFC Bank
Page: 194
Distance: 0.3785
Text: 1,493.65
 1,470.35 
 1,609.55 
 1,447.90 
 1,828.20 
Price to earnings ratio 
 21.93 
 25.23 
 28.47 
 29.48 
 17.95 
 26.40 
 22.01 
 20.31 
 16.87 
 20.71 
H1 Crore = H10 Million                    ...


In [20]:
from sklearn.cluster import KMeans
import numpy as np
import json

# load chunks with embeddings
with open("data/chunks_with_embeddings.json", "r", encoding="utf-8") as f:
    chunks = json.load(f)

# group chunks by bank
banks = {}
for chunk in chunks:
    bank = chunk["bank_name"]
    if bank not in banks:
        banks[bank] = []
    banks[bank].append(chunk)

# print count per bank
for bank, bank_chunks in banks.items():
    print(f"{bank}: {len(bank_chunks)} chunks")

Axis Bank: 1014 chunks
HDFC Bank: 1280 chunks
ICICI Bank: 757 chunks
Kotak Mahindra Bank: 1167 chunks
SBI: 1278 chunks


In [21]:
import anthropic
import numpy as np
from sklearn.cluster import KMeans
import json
import os
from dotenv import load_dotenv

load_dotenv()

client_anthropic = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))

def cluster_and_summarise(bank_name, bank_chunks, n_clusters=5):
    print(f"\nProcessing {bank_name}...")
    
    # get embeddings as matrix
    embeddings_matrix = np.array([c["embedding"] for c in bank_chunks])
    
    # cluster
    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    labels = kmeans.fit_predict(embeddings_matrix)
    
    l1_summaries = []
    
    for cluster_id in range(n_clusters):
        # get all chunks in this cluster
        cluster_chunks = [bank_chunks[i] for i in range(len(bank_chunks)) if labels[i] == cluster_id]
        
        # join texts (limit to avoid token limits)
        combined_text = "\n\n".join([c["text"] for c in cluster_chunks[:20]])
        
        print(f"  Cluster {cluster_id+1}: {len(cluster_chunks)} chunks → summarising...")
        
        # summarise with claude
        response = client_anthropic.messages.create(
            model="claude-sonnet-4-5",
            max_tokens=1000,
            messages=[{
                "role": "user",
                "content": f"""You are summarising a cluster of excerpts from {bank_name}'s FY25 annual report.
Write a 250-300 word summary covering the key financial metrics, risk factors, and strategic points in these excerpts. Be specific with numbers.

Context:
{combined_text}

Write only the summary, no preamble."""
            }]
        )
        
        summary_text = response.content[0].text
        
        l1_summaries.append({
            "chunk_id": f"l1_{bank_name.replace(' ', '_')}_{cluster_id}",
            "bank_name": bank_name,
            "cluster_id": cluster_id,
            "fiscal_year": "FY25",
            "text": summary_text,
            "level": "summary_l1",
            "source_chunk_count": len(cluster_chunks)
        })
    
    return l1_summaries

# run for all banks
all_l1_summaries = []

for bank_name, bank_chunks in banks.items():
    l1 = cluster_and_summarise(bank_name, bank_chunks)
    all_l1_summaries.extend(l1)

# save
with open("data/l1_summaries.json", "w", encoding="utf-8") as f:
    json.dump(all_l1_summaries, f, ensure_ascii=False, indent=2)

print(f"\nDone. {len(all_l1_summaries)} L1 summaries saved.")


Processing Axis Bank...
  Cluster 1: 241 chunks → summarising...
  Cluster 2: 114 chunks → summarising...
  Cluster 3: 274 chunks → summarising...
  Cluster 4: 227 chunks → summarising...
  Cluster 5: 158 chunks → summarising...

Processing HDFC Bank...
  Cluster 1: 241 chunks → summarising...
  Cluster 2: 74 chunks → summarising...
  Cluster 3: 307 chunks → summarising...
  Cluster 4: 286 chunks → summarising...
  Cluster 5: 372 chunks → summarising...

Processing ICICI Bank...
  Cluster 1: 212 chunks → summarising...
  Cluster 2: 173 chunks → summarising...
  Cluster 3: 81 chunks → summarising...
  Cluster 4: 122 chunks → summarising...
  Cluster 5: 169 chunks → summarising...

Processing Kotak Mahindra Bank...
  Cluster 1: 186 chunks → summarising...
  Cluster 2: 263 chunks → summarising...
  Cluster 3: 158 chunks → summarising...
  Cluster 4: 337 chunks → summarising...
  Cluster 5: 223 chunks → summarising...

Processing SBI...
  Cluster 1: 243 chunks → summarising...
  Cluster 2

In [22]:
def generate_l2_summaries(l1_summaries):
    # group l1 summaries by bank
    banks_l1 = {}
    for summary in l1_summaries:
        bank = summary["bank_name"]
        if bank not in banks_l1:
            banks_l1[bank] = []
        banks_l1[bank].append(summary)
    
    l2_summaries = []
    
    for bank_name, bank_l1s in banks_l1.items():
        print(f"Generating L2 summary for {bank_name}...")
        
        # combine all l1 summaries for this bank
        combined = "\n\n".join([s["text"] for s in bank_l1s])
        
        response = client_anthropic.messages.create(
            model="claude-sonnet-4-5",
            max_tokens=1500,
            messages=[{
                "role": "user",
                "content": f"""You are creating a comprehensive summary of {bank_name}'s FY25 annual report.
Below are 5 cluster summaries covering different aspects of the report.
Write a 400-500 word master summary covering the bank's overall financial performance, key metrics, risks, and strategy. Be specific with numbers.

Cluster summaries:
{combined}

Write only the summary, no preamble."""
            }]
        )
        
        l2_summaries.append({
            "chunk_id": f"l2_{bank_name.replace(' ', '_')}",
            "bank_name": bank_name,
            "fiscal_year": "FY25",
            "text": response.content[0].text,
            "level": "summary_l2"
        })
        
        print(f"  → Done")
    
    return l2_summaries

# run it
with open("data/l1_summaries.json", "r", encoding="utf-8") as f:
    l1_summaries = json.load(f)

l2_summaries = generate_l2_summaries(l1_summaries)

with open("data/l2_summaries.json", "w", encoding="utf-8") as f:
    json.dump(l2_summaries, f, ensure_ascii=False, indent=2)

print(f"\nDone. {len(l2_summaries)} L2 summaries saved.")

Generating L2 summary for Axis Bank...
  → Done
Generating L2 summary for HDFC Bank...
  → Done
Generating L2 summary for ICICI Bank...
  → Done
Generating L2 summary for Kotak Mahindra Bank...
  → Done
Generating L2 summary for SBI...
  → Done

Done. 5 L2 summaries saved.


In [23]:
# load all summaries
with open("data/l1_summaries.json", "r", encoding="utf-8") as f:
    l1_summaries = json.load(f)

with open("data/l2_summaries.json", "r", encoding="utf-8") as f:
    l2_summaries = json.load(f)

all_summaries = l1_summaries + l2_summaries
print(f"Total summaries to embed: {len(all_summaries)}")

# embed all summaries
summary_texts = [s["text"] for s in all_summaries]
summary_embeddings = embed_texts_with_retry(summary_texts, batch_size=50)

# upload to chromadb
for i, summary in enumerate(all_summaries):
    summary["embedding"] = summary_embeddings[i]

collection.add(
    ids=[s["chunk_id"] for s in all_summaries],
    embeddings=[s["embedding"] for s in all_summaries],
    documents=[s["text"] for s in all_summaries],
    metadatas=[{
        "bank_name": s["bank_name"],
        "fiscal_year": s["fiscal_year"],
        "level": s["level"]
    } for s in all_summaries]
)

print(f"\nDone. ChromaDB now has {collection.count()} total nodes.")

Total summaries to embed: 30
  Embedded 30/30 chunks...

Done. ChromaDB now has 5526 total nodes.


In [24]:
def classify_query(question):
    # override — time-sensitive keywords always go to web search
    time_keywords = ["current", "today", "now", "latest", "right now", "as of today"]
    if any(kw in question.lower() for kw in time_keywords):
        return "out_of_scope"
    
    # ranking questions always mean comparative across the 5 banks
    ranking_keywords = ["highest", "lowest", "best", "worst", "most", "least", "which bank"]
    if any(kw in question.lower() for kw in ranking_keywords):
        return "comparative"
    
    # rest of LLM classify call unchanged...
    
    response = client_anthropic.messages.create(
        model="claude-sonnet-4-5",
        max_tokens=10,
        messages=[{"role": "user", "content": f"""Classify into one of: factual / comparative / summary / out_of_scope

factual - specific metric from one named bank's FY25 annual report
comparative - comparing metrics across the 5 Indian banks (HDFC, ICICI, SBI, Axis, Kotak)
summary - broad qualitative question about a bank's strategy or approach
out_of_scope - requires live/current data after March 2025, OR completely unrelated to these 5 Indian banks
              (NOT out_of_scope: any question about ROE, NPA, CAR, NIM, profit, deposits, loans, ESG, risk
               for HDFC, ICICI, SBI, Axis Bank, or Kotak — even if no bank is named, assume it refers to these 5)

One word only. Question: {question}"""}]
    )
    
    return response.content[0].text.strip().lower()

# test all 4 query types
test_queries = [
    "What was HDFC Bank's gross NPA ratio in FY25?",
    "Compare capital adequacy ratios across all 5 banks.",
    "Summarise ICICI Bank's risk management approach.",
    "What is the RBI repo rate?",
    "What is the current RBI repo rate?",
    "What were SBI's net interest margins?",
    "Which bank had the highest return on equity?",
]

print("Testing router...\n")
for query in test_queries:
    category = classify_query(query)
    print(f"Q: {query}")
    print(f"→ {category}\n")

Testing router...

Q: What was HDFC Bank's gross NPA ratio in FY25?
→ factual

Q: Compare capital adequacy ratios across all 5 banks.
→ comparative

Q: Summarise ICICI Bank's risk management approach.
→ summary

Q: What is the RBI repo rate?
→ out_of_scope

Q: What is the current RBI repo rate?
→ out_of_scope

Q: What were SBI's net interest margins?
→ factual

Q: Which bank had the highest return on equity?
→ comparative



In [25]:
# cell 1 - imports and clients 
# to reinitailse
import anthropic
import chromadb
from openai import OpenAI
import os
import json
from dotenv import load_dotenv

load_dotenv(override=True)

client_anthropic = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))
client_openai = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# reconnect to chromadb
client_chroma = chromadb.PersistentClient(path="./chroma_db")
collection = client_chroma.get_or_create_collection(
    name="financial_rag",
    metadata={"hnsw:space": "cosine"}
)

print(f"ChromaDB has {collection.count()} nodes")
print("Clients ready ✓")

ChromaDB has 5526 nodes
Clients ready ✓


In [26]:
def embed_text(text):
    response = client_openai.embeddings.create(
        input=text,
        model="text-embedding-3-small"
    )
    return response.data[0].embedding

def hyde_retrieve(question, n_results=5):
    # generate hypothetical answer
    response = client_anthropic.messages.create(
        model="claude-sonnet-4-5",
        max_tokens=300,
        messages=[{
            "role": "user",
            "content": f"""Write a one-paragraph excerpt from an Indian bank annual report 
that would answer this question: {question}
Use specific financial numbers and banking terminology.
Respond with only the paragraph, no preamble."""
        }]
    )
    
    hypothetical_answer = response.content[0].text
    print(f"  Hypothetical answer: {hypothetical_answer[:100]}...")
    
    # embed the hypothetical answer not the question
    hyde_embedding = embed_text(hypothetical_answer)
    
    # retrieve using hypothetical embedding
    results = collection.query(
        query_embeddings=[hyde_embedding],
        n_results=n_results,
        where={"level": "leaf"},
        include=["documents", "metadatas", "distances"]
    )
    
    return results

def extract_mentioned_banks(question):
    bank_map = {
        "hdfc": "HDFC Bank",
        "icici": "ICICI Bank",
        "sbi": "SBI",
        "axis": "Axis Bank",
        "kotak": "Kotak Mahindra Bank",
        "state bank": "SBI"
    }
    q_lower = question.lower()
    mentioned = [bank for keyword, bank in bank_map.items() if keyword in q_lower]
    return mentioned if mentioned else None  # None means all banks

def multi_query_retrieve(question, n_results=10):
    mentioned_banks = extract_mentioned_banks(question)

    # build metadata filter
    if mentioned_banks and len(mentioned_banks) < 5:
        bank_filter_l2 = {
            "$and": [
                {"level": "summary_l2"},
                {"bank_name": {"$in": mentioned_banks}}
            ]
        }
        bank_filter_leaf = {"bank_name": {"$in": mentioned_banks}}
    else:
        bank_filter_l2 = {"level": "summary_l2"}
        bank_filter_leaf = None

    # pull L2 nodes (filtered to mentioned banks)
    l2_results = collection.query(
        query_embeddings=[embed_text(question)],
        n_results=5,
        where=bank_filter_l2,
        include=["documents", "metadatas", "distances"]
    )
    seen = {}
    for i in range(len(l2_results["documents"][0])):
        k = l2_results["documents"][0][i][:100]
        seen[k] = {
            "text": l2_results["documents"][0][i],
            "metadata": l2_results["metadatas"][0][i],
            "distance": l2_results["distances"][0][i]
        }

    # add leaf chunks from paraphrases (filtered to mentioned banks)
    r = client_anthropic.messages.create(
        model="claude-sonnet-4-5", max_tokens=200, temperature=0,
        messages=[{"role": "user", "content": f"Generate 3 different phrasings of this question for document retrieval:\n{question}\nNumbered list only."}]
    )
    paraphrases = [l.strip().lstrip("123.").strip() for l in r.content[0].text.strip().split("\n") if l.strip()]

    for p in paraphrases:
        query_params = dict(
            query_embeddings=[embed_text(p)],
            n_results=n_results,
            include=["documents", "metadatas", "distances"]
        )
        if bank_filter_leaf:
            query_params["where"] = {
                "$and": [{"level": "leaf"}, {"bank_name": {"$in": mentioned_banks}}]
            }
        else:
            query_params["where"] = {"level": "leaf"}

        res = collection.query(**query_params)
        for i in range(len(res["documents"][0])):
            k = res["documents"][0][i][:100]
            if k not in seen:
                seen[k] = {
                    "text": res["documents"][0][i],
                    "metadata": res["metadatas"][0][i],
                    "distance": res["distances"][0][i]
                }

    return list(seen.values())

def summary_retrieve(question, bank_name=None):
    # retrieve l2 summary nodes
    where_filter = {"level": "summary_l2"}
    if bank_name:
        where_filter = {"$and": [{"level": "summary_l2"}, {"bank_name": bank_name}]}
    
    embedding = embed_text(question)
    results = collection.query(
        query_embeddings=[embedding],
        n_results=5,
        where=where_filter,
        include=["documents", "metadatas", "distances"]
    )
    
    return results

# test all three
print("=== Testing HyDE (factual) ===")
hyde_results = hyde_retrieve("What was HDFC Bank's gross NPA ratio in FY25?")
print(f"  Retrieved {len(hyde_results['documents'][0])} chunks")
print(f"  Top result: {hyde_results['documents'][0][0][:150]}...")

print("\n=== Testing Multi-Query (comparative) ===")
mq_results = multi_query_retrieve("Compare capital adequacy ratios across all 5 banks.")
print(f"  Banks covered: {set(c['metadata']['bank_name'] for c in mq_results)}")

print("\n=== Testing Summary Retrieve ===")
sum_results = summary_retrieve("ICICI Bank risk management approach")
print(f"  Retrieved {len(sum_results['documents'][0])} summary nodes")
print(f"  Top result: {sum_results['documents'][0][0][:150]}...")

=== Testing HyDE (factual) ===
  Hypothetical answer: During FY25, HDFC Bank demonstrated robust asset quality with continued improvement in credit metric...
  Retrieved 5 chunks
  Top result: while maintaining pristine asset quality 
which has been its USP across business 
cycles. Net Proﬁt increased by 10.7 per 
cent to H67,347.4 crore on ...

=== Testing Multi-Query (comparative) ===
  Banks covered: {'Axis Bank', 'HDFC Bank', 'Kotak Mahindra Bank', 'SBI', 'ICICI Bank'}

=== Testing Summary Retrieve ===
  Retrieved 5 summary nodes
  Top result: # ICICI Bank FY25 Annual Report: Master Summary

ICICI Bank delivered robust financial performance in FY2025, with profit after tax rising 15.5% to ₹4...


In [27]:
def grade_context(question, context_text):
    response = client_anthropic.messages.create(
        model="claude-sonnet-4-5",
        max_tokens=100,
        messages=[{
            "role": "user",
            "content": f"""Given this question: {question}

And this retrieved context: {context_text[:6000]}

Rate how well the context answers the question on a scale of 0.0 to 1.0.
0.0 = context is completely irrelevant.
0.5 = context partially answers the question.
0.8 = context mostly answers the question with minor gaps.
1.0 = context fully answers the question.

Important: if context contains relevant financial data even if incomplete, score at least 0.6.

Respond with only: score|one-sentence rationale
Example: 0.8|Context contains CAR data for most banks."""
        }]
    )
    
    result = response.content[0].text.strip()
    parts = result.split("|")
    score = float(parts[0].strip())
    rationale = parts[1].strip() if len(parts) > 1 else "No rationale"
    return score, rationale


def retrieve_by_type(question, query_type, top_k=5):
    if query_type == "factual":
        results = hyde_retrieve(question, n_results=top_k)
        chunks = []
        for i in range(len(results["documents"][0])):
            chunks.append({
                "text": results["documents"][0][i],
                "metadata": results["metadatas"][0][i]
            })
        return chunks
    
    elif query_type == "comparative":
        return multi_query_retrieve(question, n_results=top_k)
    
    elif query_type == "summary":
        results = summary_retrieve(question)
        chunks = []
        for i in range(len(results["documents"][0])):
            chunks.append({
                "text": results["documents"][0][i],
                "metadata": results["metadatas"][0][i]
            })
        return chunks
    
    else:
        return []


def generate_answer(question, chunks):
    context_parts = []
    for c in chunks:
        level = c['metadata']['level']
        bank = c['metadata']['bank_name']
        page = c['metadata'].get('page_number', 'N/A')
        
        # only show page for leaf chunks
        if level == 'leaf':
            citation = f"[{bank}, Page {page}]"
        else:
            citation = f"[{bank}, {level}]"
        
        context_parts.append(f"{citation}\n{c['text']}")
    
    context = "\n\n".join(context_parts)
    response = client_anthropic.messages.create(
        model="claude-sonnet-4-5",
        max_tokens=1000,
        temperature=0,
        messages=[{"role": "user", "content": f"""You are a financial analyst assistant. Answer only based on the provided context.
Rules:
1. For every fact or number, cite the source as [Bank Name, Page X].
2. These are FY25 annual reports — assume all figures are FY25 unless stated otherwise.
3. If the context contains relevant data even without explicit FY25 label, use it and cite it.
4. Only say 'Not found in provided reports' if the context has absolutely no relevant information.
5. Never say 'not found' and then quote relevant data in the same response.

Context:
{context}

Question: {question}"""}]
    )
    return response.content[0].text
    

def query_with_grading(question):
    print(f"\nQuestion: {question}")
    
    query_type = classify_query(question)
    print(f"Router: {query_type}")
    
    # out_of_scope → web search directly
    if query_type == "out_of_scope":
        return web_search_fallback(question)
    
    top_k = 15 if query_type == "comparative" else 5
    chunks = retrieve_by_type(question, query_type, top_k=top_k +10)
    # sort for consistency
    chunks = sorted(chunks, key=lambda x: x["metadata"]["bank_name"])
    context_text = " ".join([c["text"] for c in chunks])
    
    score, rationale = grade_context(question, context_text)
    print(f"Grade: {score} — {rationale}")
    
    if score < 0.4:
        print(f"Score too low — re-retrieving...")
        chunks = retrieve_by_type(question, query_type, top_k=top_k + 8)
        context_text = " ".join([c["text"] for c in chunks])
        score, rationale = grade_context(question, context_text)
        print(f"New grade: {score} — {rationale}")
    
    # still insufficient → web search fallback
    if score < 0.3:
        print("Still insufficient — falling back to web search...")
        return web_search_fallback(question)
    
    answer = generate_answer(question, chunks)
    sources = list({
        (c["metadata"]["bank_name"], c["metadata"].get("page_number", "N/A"))
        for c in chunks
    })
    
    return {
        "answer": answer,
        "confidence": score,
        "query_type": query_type,
        "sources": sources
    }

In [ ]:
# result = query_with_grading("Compare capital adequacy ratios across all 5 banks.")
# print(f"\nFull Answer:\n{result['answer']}")
# print(f"\nConfidence: {result['confidence']}")
# print(f"Sources: {result['sources']}")

In [ ]:
# result = hyde_retrieve("Who is the CEO of SBI?", n_results=5)
# for i in range(len(result["documents"][0])):
#     print(f"\n--- Chunk {i+1} ---")
#     print(f"Bank: {result['metadatas'][0][i]['bank_name']}")
#     print(f"Page: {result['metadatas'][0][i]['page_number']}")
#     print(f"Text: {result['documents'][0][i][:300]}")

In [ ]:
# from tavily import TavilyClient
# import os
# from dotenv import load_dotenv

# load_dotenv(override=True)

# tavily_client = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))

# # quick test
# results = tavily_client.search(query="RBI repo rate 2025", search_depth="basic", max_results=3)
# print(results["results"][0]["content"])

In [ ]:
# result = query_with_grading("What is the current RBI repo rate?")
# print(result["answer"])
# print(result["sources"])

In [ ]:
# results = tavily_client.search(query="RBI repo rate 2025", search_depth="advanced", max_results=3)
# print(results["results"][0]["content"])

In [ ]:
from supabase import create_client
import os
from dotenv import load_dotenv

load_dotenv(override=True)

sb = create_client(os.getenv("SUPABASE_URL"), os.getenv("SUPABASE_KEY"))

# try inserting a test row
result = sb.table("chat_history").insert({
    "question": "test question",
    "query_type": "factual",
    "confidence": 0.9,
    "answer": "test answer",
    "web_search_used": False,
    "sources": "HDFC pg38"
}).execute()

print(result)

data=[{'id': 'ce48bfb9-b9b8-4f66-87eb-304c2f38ab03', 'timestamp': '2026-06-06T05:42:33.033827+00:00', 'question': 'test question', 'query_type': 'factual', 'confidence': 0.9, 'answer': 'test answer', 'web_search_used': False, 'sources': 'HDFC pg38'}] count=None


In [1]:
import sys
!{sys.executable} -m pip uninstall pinecone-client -y
!{sys.executable} -m pip install pinecone

Found existing installation: pinecone-client 6.0.0
Uninstalling pinecone-client-6.0.0:
  Successfully uninstalled pinecone-client-6.0.0
  Using cached pinecone-9.1.0-cp310-abi3-win_amd64.whl.metadata (6.3 kB)
  Using cached msgspec-0.21.1-cp313-cp313-win_amd64.whl.metadata (5.9 kB)
Using cached pinecone-9.1.0-cp310-abi3-win_amd64.whl (2.6 MB)
Using cached msgspec-0.21.1-cp313-cp313-win_amd64.whl (189 kB)


In [4]:
from pinecone import Pinecone
import json
import os
from dotenv import load_dotenv

load_dotenv(override=True)

# connect to pinecone
pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))
index = pc.Index("financial-rag")

# ── wipe existing index before re-uploading ──
index.delete(delete_all=True)
stats = index.describe_index_stats()
print(f"Index wiped. Vector count: {stats['total_vector_count']}")  # expect 0

# load chunks with embeddings
with open("data/chunks_with_embeddings.json", "r", encoding="utf-8") as f:
    chunks = json.load(f)

# load l1 and l2 summaries
with open("data/l1_summaries.json", "r", encoding="utf-8") as f:
    l1_summaries = json.load(f)

with open("data/l2_summaries.json", "r", encoding="utf-8") as f:
    l2_summaries = json.load(f)

all_nodes = chunks + l1_summaries + l2_summaries
print(f"Total nodes to upload: {len(all_nodes)}")

# upload in batches of 100
batch_size = 100
total = len(all_nodes)

for i in range(0, total, batch_size):
    batch = all_nodes[i:i+batch_size]
    
    vectors = []
    for node in batch:
        # skip nodes without embeddings
        if "embedding" not in node:
            continue
        vectors.append({
            "id": node["chunk_id"],
            "values": node["embedding"],
            "metadata": {
                "bank_name": node["bank_name"],
                "page_number": node.get("page_number", -1),
                "fiscal_year": node.get("fiscal_year", "FY25"),
                "level": node["level"],
                "text": node["text"][:1000]  # pinecone metadata limit
            }
        })
    
    if vectors:
        index.upsert(vectors=vectors)
        print(f"Uploaded {min(i+batch_size, total)}/{total} nodes...")

print(f"\nDone. Verifying...")
stats = index.describe_index_stats()
print(f"Pinecone index has {stats['total_vector_count']} vectors")

Index wiped. Vector count: 0
Total nodes to upload: 5526
Uploaded 100/5526 nodes...
Uploaded 200/5526 nodes...
Uploaded 300/5526 nodes...
Uploaded 400/5526 nodes...
Uploaded 500/5526 nodes...
Uploaded 600/5526 nodes...
Uploaded 700/5526 nodes...
Uploaded 800/5526 nodes...
Uploaded 900/5526 nodes...
Uploaded 1000/5526 nodes...
Uploaded 1100/5526 nodes...
Uploaded 1200/5526 nodes...
Uploaded 1300/5526 nodes...
Uploaded 1400/5526 nodes...
Uploaded 1500/5526 nodes...
Uploaded 1600/5526 nodes...
Uploaded 1700/5526 nodes...
Uploaded 1800/5526 nodes...
Uploaded 1900/5526 nodes...
Uploaded 2000/5526 nodes...
Uploaded 2100/5526 nodes...
Uploaded 2200/5526 nodes...
Uploaded 2300/5526 nodes...
Uploaded 2400/5526 nodes...
Uploaded 2500/5526 nodes...
Uploaded 2600/5526 nodes...
Uploaded 2700/5526 nodes...
Uploaded 2800/5526 nodes...
Uploaded 2900/5526 nodes...
Uploaded 3000/5526 nodes...
Uploaded 3100/5526 nodes...
Uploaded 3200/5526 nodes...
Uploaded 3300/5526 nodes...
Uploaded 3400/5526 nodes...


In [5]:
# embed and upload l1 and l2 summaries
from openai import OpenAI
client_openai = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

all_summaries = l1_summaries + l2_summaries
print(f"Embedding {len(all_summaries)} summary nodes...")

texts = [s["text"] for s in all_summaries]
response = client_openai.embeddings.create(
    input=texts,
    model="text-embedding-3-small"
)
embeddings = [item.embedding for item in response.data]

vectors = []
for i, summary in enumerate(all_summaries):
    vectors.append({
        "id": summary["chunk_id"],
        "values": embeddings[i],
        "metadata": {
            "bank_name": summary["bank_name"],
            "page_number": -1,
            "fiscal_year": summary.get("fiscal_year", "FY25"),
            "level": summary["level"],
            "text": summary["text"][:1000]
        }
    })

index.upsert(vectors=vectors)
print(f"Uploaded {len(vectors)} summary nodes")

stats = index.describe_index_stats()
print(f"Pinecone index now has {stats['total_vector_count']} vectors")

Embedding 30 summary nodes...
Uploaded 30 summary nodes
Pinecone index now has 5526 vectors
